# Issue #1 — Thiết Lập Nền Tảng Mô Hình
**Dự án:** Corporación Favorita Store Sales · Ecuador 2013–2017 · Metric: **RMSLE**

In [1]:
import os
import numpy as np
import pandas as pd

BASE       = r"D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series"
TRAIN_PATH = os.path.join(BASE, "data", "processed", "train_final.csv")
TEST_PATH  = os.path.join(BASE, "data", "processed", "test_final.csv")
SPLITS_DIR = os.path.join(BASE, "data", "processed", "splits")
MODELS_DIR = os.path.join(BASE, "models")

os.makedirs(SPLITS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

## Task 1 — Tổng Quan Dữ Liệu
Xác nhận dữ liệu sạch (zero NaN) và đầy đủ 6 nhóm feature trước khi xây dựng mô hình.

In [2]:
df = pd.read_csv(TRAIN_PATH, parse_dates=["date"])

sales    = df["sales"]
zero_pct = (sales == 0).mean() * 100

print(f"Shape: {df.shape} | Date: {df['date'].min().date()} → {df['date'].max().date()} | Stores: {df['store_nbr'].nunique()} | Families: {df['family'].nunique()}")
print(f"Sales — mean: {sales.mean():.1f} | median: {sales.median():.1f} | std: {sales.std():.1f} | zeros: {zero_pct:.1f}% | skew: {sales.skew():.3f}")

nan_total = df.isnull().sum().sum()
if nan_total > 0:
    print(f"NaN count: {nan_total}")

FEATURE_GROUPS = {
    "temporal": ["year","month","day","day_of_week","week_of_year","quarter",
                 "is_weekend","is_month_start","is_month_end","is_payday"],
    "lag":      ["lag_1","lag_2","lag_3","lag_7","lag_14","lag_28","lag_364"],
    "rolling":  ["rolling_mean_7","rolling_mean_14","rolling_mean_28","rolling_std_7"],
    "holiday":  ["is_national_holiday","is_regional_holiday","is_local_holiday",
                 "is_transferred_holiday","holiday_type","is_carnaval_feature",
                 "days_to_next_holiday","days_after_last_holiday","is_earthquake_period"],
    "oil":      ["oil_price","oil_price_lag_7","oil_price_lag_14",
                 "oil_price_rolling_mean_28","oil_price_change_pct"],
    "encoding": ["store_type_enc","city_freq","state_freq","cluster",
                 "store_type_encoded","store_family_te","onpromotion"],
}

for group, cols in FEATURE_GROUPS.items():
    missing = [c for c in cols if c not in df.columns]
    status  = "OK" if not missing else f"MISSING {missing}"
    print(f"  {group:10s} ({len(cols):2d} cols): {status}")

Shape: (2830221, 51) | Date: 2013-01-01 → 2017-05-12 | Stores: 54 | Families: 33
Sales — mean: 350.3 | median: 10.0 | std: 1086.2 | zeros: 32.3% | skew: 7.546
  temporal   (10 cols): OK
  lag        ( 7 cols): OK
  rolling    ( 4 cols): OK
  holiday    ( 9 cols): OK
  oil        ( 5 cols): OK
  encoding   ( 7 cols): OK


## Task 2 — Chia Tập Theo Thời Gian
`train_final.csv` kết thúc 2017-05-12. Dùng 15 ngày cuối làm validation. Test set là holdout cuộc thi (không có nhãn `sales`).

In [3]:
TRAIN_END = "2017-04-27"
VAL_START = "2017-04-28"
VAL_END   = "2017-05-12"

train_df = df[df["date"] <= TRAIN_END].copy()
val_df   = df[(df["date"] >= VAL_START) & (df["date"] <= VAL_END)].copy()
test_raw = pd.read_csv(TEST_PATH, parse_dates=["date"])

assert train_df["date"].max() < val_df["date"].min(), "DATA LEAK: train/val overlap"

print(f"Train      : {train_df.shape} | {train_df['date'].min().date()} → {train_df['date'].max().date()}")
print(f"Validation : {val_df.shape}   | {val_df['date'].min().date()} → {val_df['date'].max().date()}")
print(f"Test       : {test_raw.shape} | {test_raw['date'].min().date()} → {test_raw['date'].max().date()} (no sales labels)")

Train      : (2804868, 51) | 2013-01-01 → 2017-04-27
Validation : (25353, 51)   | 2017-04-28 → 2017-05-12
Test       : (28512, 7) | 2017-08-16 → 2017-08-31 (no sales labels)


## Task 3 — Log-Transform Biến Mục Tiêu
Dùng `log1p` thay vì `log` để xử lý an toàn 32% giá trị zero. Giữ nguyên thang gốc để tính RMSLE.

In [4]:
for name, split in [("Train", train_df), ("Validation", val_df)]:
    raw  = split["sales"]
    tran = np.log1p(raw)
    print(f"{name} BEFORE log1p: mean={raw.mean():.2f}, median={raw.median():.2f}, skew={raw.skew():.3f}")
    print(f"{name} AFTER  log1p: mean={tran.mean():.4f}, median={tran.median():.4f}, skew={tran.skew():.4f}")

train_target_log = np.log1p(train_df["sales"])
val_target_log   = np.log1p(val_df["sales"])
val_target_orig  = val_df["sales"].copy()

Train BEFORE log1p: mean=348.93, median=10.00, skew=7.574
Train AFTER  log1p: mean=2.8772, median=2.3979, skew=0.4290
Validation BEFORE log1p: mean=505.90, median=32.00, skew=5.571
Validation AFTER  log1p: mean=3.6851, median=3.4965, skew=0.1965


## Task 4 — Xác Định Feature Columns
Loại bỏ: `date`, `sales`, `id`, `family`, `type`, `city`, `state`, `store_family` — hoặc là target, hoặc đã được encode.

In [5]:
EXCLUDE      = {"date", "sales", "id", "family", "type", "city", "state", "store_family"}
feature_cols = [c for c in df.columns if c not in EXCLUDE]

grouped_all = set()
for group, cols in FEATURE_GROUPS.items():
    in_feat = [c for c in cols if c in feature_cols]
    grouped_all.update(in_feat)
    print(f"  {group:10s} ({len(in_feat):2d}): {in_feat}")

other = [c for c in feature_cols if c not in grouped_all]
if other:
    print(f"  {'other':10s} ({len(other):2d}): {other}")

print(f"\nTotal features: {len(feature_cols)}")

  temporal   (10): ['year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_payday']
  lag        ( 7): ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_364']
  rolling    ( 4): ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']
  holiday    ( 9): ['is_national_holiday', 'is_regional_holiday', 'is_local_holiday', 'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature', 'days_to_next_holiday', 'days_after_last_holiday', 'is_earthquake_period']
  oil        ( 5): ['oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct']
  encoding   ( 7): ['store_type_enc', 'city_freq', 'state_freq', 'cluster', 'store_type_encoded', 'store_family_te', 'onpromotion']
  other      ( 1): ['store_nbr']

Total features: 43


## Task 5 — Lưu Artifacts
Các file này được dùng trực tiếp bởi Issue #2 (ARIMA), #3 (LightGBM), #4 (XGBoost), #5 (Ensemble).

In [6]:
train_X    = train_df[feature_cols].reset_index(drop=True)
val_X      = val_df[feature_cols].reset_index(drop=True)
train_y    = train_target_log.reset_index(drop=True).rename("sales_log")
val_y      = val_target_log.reset_index(drop=True).rename("sales_log")
val_y_orig = val_target_orig.reset_index(drop=True).rename("sales")

test_feat_cols = [c for c in feature_cols if c in test_raw.columns]
test_X = test_raw[test_feat_cols].reset_index(drop=True)

saves = [
    (os.path.join(SPLITS_DIR, "train_features.csv"),      train_X),
    (os.path.join(SPLITS_DIR, "train_target.csv"),        train_y.to_frame()),
    (os.path.join(SPLITS_DIR, "val_features.csv"),        val_X),
    (os.path.join(SPLITS_DIR, "val_target.csv"),          val_y.to_frame()),
    (os.path.join(SPLITS_DIR, "val_target_original.csv"), val_y_orig.to_frame()),
    (os.path.join(SPLITS_DIR, "test_features.csv"),       test_X),
]

for path, data in saves:
    data.to_csv(path, index=False)
    print(f"Saved: {os.path.basename(path)} {data.shape}")

feat_txt = os.path.join(MODELS_DIR, "feature_columns.txt")
with open(feat_txt, "w") as f:
    f.write("\n".join(feature_cols))
print(f"Saved: feature_columns.txt ({len(feature_cols)} features)")

pd.DataFrame([{
    "train": f"{train_df['date'].min().date()} → {train_df['date'].max().date()} ({len(train_df):,} rows)",
    "val":   f"{val_df['date'].min().date()} → {val_df['date'].max().date()} ({len(val_df):,} rows)",
    "test":  f"{test_raw['date'].min().date()} → {test_raw['date'].max().date()} ({len(test_raw):,} rows)",
    "features": len(feature_cols),
    "target_transform": "log1p",
    "eval_metric": "RMSLE",
}]).to_csv(os.path.join(SPLITS_DIR, "split_metadata.csv"), index=False)
print("Saved: split_metadata.csv")

Saved: train_features.csv (2804868, 43)
Saved: train_target.csv (2804868, 1)
Saved: val_features.csv (25353, 43)
Saved: val_target.csv (25353, 1)
Saved: val_target_original.csv (25353, 1)
Saved: test_features.csv (28512, 3)
Saved: feature_columns.txt (43 features)
Saved: split_metadata.csv
